
# Multi-maturity deterministic arbitrage detection

This Colab notebook extends the one-maturity arbitrage-search code to a multi-maturity option market.

The terminal state is now a path vector

$$
s=(s_1,\ldots,s_m) \in \mathbb{R}_+^m,
$$

where \(s_j\) is the underlying value at maturity \(T_j\). All cashflows are accumulated to the common horizon

$$
\bar T = \max_j T_j.
$$

A payoff received at maturity \(T_j\) is multiplied by

$$
A_j = \exp(r(\bar T-T_j)).
$$

For each maturity, calls, puts, and a maturity-specific stock liquidation position enter the static portfolio. The deterministic no-arbitrage condition is pathwise:

$$
G_\theta(s) \geq 0 \quad \text{for every } s\in\mathbb{R}_+^m,
$$

with strict positivity at least somewhere. Since the payoff is separately piecewise affine in each coordinate, the global no-loss condition is reduced to finitely many inequalities on the Cartesian grid

$$
V = X_1\times\cdots\times X_m,
\qquad
X_j = \{0\}\cup\{\text{strikes traded at maturity }T_j\},
$$

together with one nonnegative right-tail slope condition per maturity.

The notebook contains:

1. a synthetic Black--Scholes example,
2. a general solver for multi-maturity deterministic arbitrage,
3. optional Yahoo Finance helpers for Colab,
4. diagnostic tables and payoff-slice plots.

Because the Cartesian grid grows as \(\prod_j |X_j|\), the Yahoo helper deliberately keeps only a limited number of strikes around the current spot price.


## Cell 1 — Install the required packages

This cell installs the numerical, plotting, and Yahoo Finance packages used by the notebook.

In [ ]:
# Install the external packages required by the notebook.
# This cell is intentionally small so that it is easy to re-run in Colab.
%pip -q install numpy pandas scipy matplotlib yfinance prettytable


## Cell 2 — Load the multi-maturity arbitrage-search implementation

This cell defines the synthetic data generator, Yahoo helpers, finite Cartesian grid construction, LP/MILP solver, diagnostic tables, and payoff-slice plots.

In [ ]:

# ============================================================
# Multi-maturity deterministic arbitrage detection
# ============================================================
#
# This Colab-ready code extends the one-maturity deterministic
# arbitrage search to a multi-maturity option market.
#
# The state is a path vector
#
#     s = (s_1, ..., s_m),
#
# where s_j is the underlying value at maturity T_j.
#
# All cashflows are accumulated to the common horizon T_bar.
# A payoff paid at maturity T_j is multiplied by
#
#     A_j = exp(r * (T_bar - T_j)).
#
# The solver searches for a zero-cost static portfolio whose
# accumulated payoff is non-negative for every path vector and
# strictly positive at at least one finite certificate or in one
# right-tail direction.
#
# ============================================================

from __future__ import annotations

import itertools
import math
from dataclasses import dataclass
from datetime import date, datetime, timezone
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.stats import norm


# ============================================================
# Data containers
# ============================================================

@dataclass
class MultiMaturityMarket:
    """
    Container for a multi-maturity option market.

    Attributes
    ----------
    S0 : float
        Current underlying price.
    r : float
        Continuously compounded risk-free rate.
    maturities : list of float
        Times to maturity in years.
    chains : list of dict
        One dictionary per maturity. Each dictionary contains
        call and put DataFrames with columns:
            strike, bid, ask.
    stock_bid : float
        Effective bid for one share of the underlying.
    stock_ask : float
        Effective ask for one share of the underlying.
    name : str
        Optional market name used in printed output.
    """

    S0: float
    r: float
    maturities: List[float]
    chains: List[Dict[str, pd.DataFrame]]
    stock_bid: Optional[float] = None
    stock_ask: Optional[float] = None
    name: str = "multi-maturity market"

    def __post_init__(self):
        if self.stock_bid is None:
            self.stock_bid = float(self.S0)
        if self.stock_ask is None:
            self.stock_ask = float(self.S0)
        self.S0 = float(self.S0)
        self.r = float(self.r)
        self.maturities = [float(t) for t in self.maturities]
        if len(self.maturities) != len(self.chains):
            raise ValueError("maturities and chains must have the same length.")
        if any(t <= 0 for t in self.maturities):
            raise ValueError("All maturities must be positive.")
        if not np.all(np.diff(self.maturities) >= 0):
            raise ValueError("Maturities must be sorted in increasing order.")


# ============================================================
# Black-Scholes prices for synthetic tests
# ============================================================

def bs_call_put(S: float, K, r: float, sigma: float, tau: float):
    """
    Compute Black-Scholes European call and put prices.

    This function is used only to generate synthetic examples.
    The arbitrage detector itself does not assume Black-Scholes.
    """
    K = np.asarray(K, dtype=float)
    S = float(S)
    r = float(r)
    sigma = float(sigma)
    tau = float(tau)

    if tau <= 0:
        call = np.maximum(S - K, 0.0)
        put = np.maximum(K - S, 0.0)
        return call, put

    vol_sqrt = sigma * np.sqrt(tau)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / vol_sqrt
    d2 = d1 - vol_sqrt
    call = S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    put = K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return call, put


def make_synthetic_multi_maturity_market(
    S0: float = 100.0,
    r: float = 0.03,
    sigma: float = 0.20,
    maturities: Sequence[float] = (0.25, 0.50, 1.00),
    spread_rate: float = 0.03,
    absolute_spread: float = 0.05,
    seed: int = 123,
    inject_violation: bool = True,
) -> MultiMaturityMarket:
    """
    Create a synthetic multi-maturity bid-ask option market.

    The quotes are generated from Black-Scholes mid prices and
    then converted to bid-ask intervals. If inject_violation=True,
    a few quotes are deliberately distorted so that the detector
    has a meaningful test case.
    """
    rng = np.random.default_rng(seed)
    chains = []

    base_strikes = np.arange(80.0, 121.0, 10.0)

    for j, tau in enumerate(maturities):
        K = base_strikes.copy()
        call_mid, put_mid = bs_call_put(S0, K, r, sigma, tau)

        # Small maturity-dependent noise makes the example less artificial.
        call_mid = call_mid + rng.normal(0.0, 0.05, len(K))
        put_mid = put_mid + rng.normal(0.0, 0.05, len(K))
        call_mid = np.maximum(call_mid, 0.01)
        put_mid = np.maximum(put_mid, 0.01)

        call_spread = spread_rate * np.maximum(call_mid, 1.0) + absolute_spread
        put_spread = spread_rate * np.maximum(put_mid, 1.0) + absolute_spread

        call_bid = np.maximum(call_mid - 0.5 * call_spread, 0.0)
        call_ask = call_mid + 0.5 * call_spread
        put_bid = np.maximum(put_mid - 0.5 * put_spread, 0.0)
        put_ask = put_mid + 0.5 * put_spread

        # Deliberately inject a few bid-ask and static distortions.
        if inject_violation and j == 1 and len(K) >= 5:
            call_bid[2] = call_ask[2] + 0.40
            put_bid[3] = put_ask[3] + 0.35
        if inject_violation and j == len(maturities) - 1 and len(K) >= 5:
            call_bid[1] += 0.75
            put_ask[4] = max(put_ask[4] - 0.70, 0.0)

        calls = pd.DataFrame({"strike": K, "bid": call_bid, "ask": call_ask})
        puts = pd.DataFrame({"strike": K, "bid": put_bid, "ask": put_ask})
        chains.append({"calls": calls, "puts": puts})

    return MultiMaturityMarket(
        S0=S0,
        r=r,
        maturities=list(maturities),
        chains=chains,
        stock_bid=S0,
        stock_ask=S0,
        name="synthetic multi-maturity market",
    )


# ============================================================
# Quote cleaning and Yahoo helpers
# ============================================================

def _clean_option_side(df: pd.DataFrame, max_rows: Optional[int], S0: float) -> pd.DataFrame:
    """
    Clean one option side returned by Yahoo Finance.

    The routine keeps strike, bid, ask, and lastPrice if available.
    Missing or non-positive asks are replaced by lastPrice when
    possible. Missing bids are replaced by zero.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask"])

    out = df.copy()
    for col in ["strike", "bid", "ask", "lastPrice"]:
        if col not in out.columns:
            out[col] = np.nan

    out = out[["strike", "bid", "ask", "lastPrice"]].copy()
    out = out.dropna(subset=["strike"])
    out["strike"] = out["strike"].astype(float)
    out["bid"] = pd.to_numeric(out["bid"], errors="coerce").fillna(0.0)
    out["ask"] = pd.to_numeric(out["ask"], errors="coerce")
    out["lastPrice"] = pd.to_numeric(out["lastPrice"], errors="coerce")

    ask_bad = out["ask"].isna() | (out["ask"] <= 0)
    out.loc[ask_bad, "ask"] = out.loc[ask_bad, "lastPrice"]
    out["ask"] = out["ask"].fillna(out["bid"])
    out["ask"] = np.maximum(out["ask"].to_numpy(dtype=float), out["bid"].to_numpy(dtype=float))

    out = out[(out["strike"] > 0) & np.isfinite(out["ask"])]
    out = out.sort_values("strike").reset_index(drop=True)

    if max_rows is not None and len(out) > max_rows:
        out["moneyness_distance"] = np.abs(out["strike"] / S0 - 1.0)
        out = out.sort_values("moneyness_distance").head(max_rows)
        out = out.sort_values("strike").drop(columns=["moneyness_distance"])
        out = out.reset_index(drop=True)

    return out[["strike", "bid", "ask"]]


def apply_transaction_cost_perturbation(
    market: MultiMaturityMarket,
    perturbation_percent: float = 0.0,
    absolute_perturbation: float = 0.0,
) -> MultiMaturityMarket:
    """
    Widen all bid-ask spreads conservatively.

    Effective ask = ask * (1 + p/100) + absolute_perturbation.
    Effective bid = max(0, bid * (1 - p/100) - absolute_perturbation).
    """
    p = float(perturbation_percent) / 100.0
    absolute_perturbation = float(absolute_perturbation)
    new_chains = []

    for chain in market.chains:
        new_chain = {}
        for side in ["calls", "puts"]:
            df = chain[side].copy()
            df["ask"] = df["ask"].astype(float) * (1.0 + p) + absolute_perturbation
            df["bid"] = np.maximum(0.0, df["bid"].astype(float) * (1.0 - p) - absolute_perturbation)
            df["ask"] = np.maximum(df["ask"], df["bid"])
            new_chain[side] = df
        new_chains.append(new_chain)

    return MultiMaturityMarket(
        S0=market.S0,
        r=market.r,
        maturities=market.maturities,
        chains=new_chains,
        stock_bid=max(0.0, market.stock_bid * (1.0 - p) - absolute_perturbation),
        stock_ask=market.stock_ask * (1.0 + p) + absolute_perturbation,
        name=market.name + " with perturbed bid-ask spreads",
    )


def fetch_yahoo_multi_maturity_market(
    ticker: str,
    expiration_dates: Sequence[str],
    r: float = 0.03,
    max_options_per_side: int = 6,
) -> MultiMaturityMarket:
    """
    Download a small multi-maturity option market from Yahoo Finance.

    To avoid a Cartesian-grid explosion, the helper keeps only the
    strikes closest to the current spot price on each option side.
    Increase max_options_per_side only when the number of maturities
    is small.
    """
    import yfinance as yf

    tk = yf.Ticker(ticker)
    hist = tk.history(period="5d")
    if hist.empty:
        raise RuntimeError("Could not download a recent stock price from Yahoo Finance.")

    S0 = float(hist["Close"].dropna().iloc[-1])
    stock_bid = S0
    stock_ask = S0

    today = datetime.now(timezone.utc).date()
    maturities = []
    chains = []

    for exp in expiration_dates:
        exp_date = datetime.strptime(exp, "%Y-%m-%d").date()
        tau = max((exp_date - today).days / 365.25, 1.0 / 365.25)
        opt = tk.option_chain(exp)
        calls = _clean_option_side(opt.calls, max_options_per_side, S0)
        puts = _clean_option_side(opt.puts, max_options_per_side, S0)
        if len(calls) == 0 and len(puts) == 0:
            continue
        maturities.append(tau)
        chains.append({"calls": calls, "puts": puts})

    if not chains:
        raise RuntimeError("No option chains were downloaded.")

    order = np.argsort(maturities)
    maturities = [maturities[i] for i in order]
    chains = [chains[i] for i in order]

    return MultiMaturityMarket(
        S0=S0,
        r=r,
        maturities=maturities,
        chains=chains,
        stock_bid=stock_bid,
        stock_ask=stock_ask,
        name=f"Yahoo market for {ticker.upper()}",
    )


def show_yahoo_expirations(ticker: str):
    """Print the expiration dates currently available from Yahoo Finance."""
    import yfinance as yf
    tk = yf.Ticker(ticker)
    expirations = list(tk.options)
    if not expirations:
        raise RuntimeError("Yahoo Finance did not return expiration dates for this ticker.")
    print(f"Available expirations for {ticker.upper()}:")
    for k, exp in enumerate(expirations, start=1):
        print(f"{k:2d}. {exp}")
    return expirations


# ============================================================
# Finite grid construction
# ============================================================

def maturity_strikes(chain: Dict[str, pd.DataFrame]) -> np.ndarray:
    """
    Return the sorted union of call and put strikes for one maturity.
    """
    values = []
    for side in ["calls", "puts"]:
        if side in chain and len(chain[side]) > 0:
            values.extend(chain[side]["strike"].astype(float).tolist())
    if not values:
        return np.array([], dtype=float)
    return np.array(sorted(set(np.round(values, 10))), dtype=float)


def build_state_grid(market: MultiMaturityMarket, max_grid_points: int = 20000):
    """
    Build the Cartesian certificate grid V = X_1 x ... x X_m.

    Each X_j contains zero and all strikes traded at maturity T_j.
    The product size can grow quickly, so max_grid_points is used
    as a safety limit.
    """
    node_sets = []
    for chain in market.chains:
        K = maturity_strikes(chain)
        xj = np.concatenate(([0.0], K))
        node_sets.append(xj)

    grid_size = int(np.prod([len(x) for x in node_sets]))
    if grid_size > max_grid_points:
        raise ValueError(
            f"The Cartesian grid has {grid_size} points, which exceeds max_grid_points={max_grid_points}. "
            "Reduce the number of maturities or the number of strikes per maturity."
        )

    vertices = np.array(list(itertools.product(*node_sets)), dtype=float)
    return node_sets, vertices


def _option_rows(chain: Dict[str, pd.DataFrame], side: str) -> pd.DataFrame:
    """Return a normalized option DataFrame for one side."""
    df = chain.get(side, pd.DataFrame(columns=["strike", "bid", "ask"])).copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["strike", "bid", "ask"])
    df = df[["strike", "bid", "ask"]].copy()
    df["strike"] = df["strike"].astype(float)
    df["bid"] = df["bid"].astype(float)
    df["ask"] = df["ask"].astype(float)
    return df.reset_index(drop=True)


# ============================================================
# Main multi-maturity arbitrage solver
# ============================================================

def build_and_solve_multi_maturity_arbitrage(
    market: MultiMaturityMarket,
    initial_capital: float = 0.0,
    stock_bound: float = 10.0,
    option_bound: float = 10.0,
    bank_bound: float = 1.0e6,
    eta_tail: float = 1.0,
    integer_options: bool = False,
    max_grid_points: int = 20000,
    time_limit: Optional[float] = None,
    mip_rel_gap: Optional[float] = None,
    tol: float = 1e-8,
) -> Dict:
    """
    Search for a multi-maturity deterministic arbitrage portfolio.

    The optimization problem is a bounded LP/MILP. Option positions
    are represented as long and short quantities because bid-ask
    execution is asymmetric.

    The objective maximizes

        sum_v D_v + eta_tail * sum_j M_j,

    where D_v is a payoff margin at a finite path certificate and
    M_j is a right-tail slope margin for maturity j.
    """
    m = len(market.maturities)
    T_bar = max(market.maturities)
    A = np.exp(market.r * (T_bar - np.asarray(market.maturities, dtype=float)))
    df_bar_growth = math.exp(market.r * T_bar)

    node_sets, vertices = build_state_grid(market, max_grid_points=max_grid_points)
    L = len(vertices)

    calls = [_option_rows(chain, "calls") for chain in market.chains]
    puts = [_option_rows(chain, "puts") for chain in market.chains]

    # --------------------------------------------------------
    # Variable indexing
    # --------------------------------------------------------
    # z contains:
    #   bank b
    #   stock long and short quantities per maturity
    #   call long and short quantities for every listed call
    #   put long and short quantities for every listed put
    #   vertex payoff margins D_v
    #   tail-slope margins M_j
    # --------------------------------------------------------

    idx = {}
    start = 0

    idx["bank"] = start
    start += 1

    idx["stock_long"] = slice(start, start + m)
    start += m
    idx["stock_short"] = slice(start, start + m)
    start += m

    idx["call_long"] = []
    idx["call_short"] = []
    for j in range(m):
        n = len(calls[j])
        idx["call_long"].append(slice(start, start + n))
        start += n
        idx["call_short"].append(slice(start, start + n))
        start += n

    idx["put_long"] = []
    idx["put_short"] = []
    for j in range(m):
        n = len(puts[j])
        idx["put_long"].append(slice(start, start + n))
        start += n
        idx["put_short"].append(slice(start, start + n))
        start += n

    idx["D"] = slice(start, start + L)
    start += L
    idx["M"] = slice(start, start + m)
    start += m

    num_vars = start

    # --------------------------------------------------------
    # Objective: scipy.milp minimizes, so use negative margins.
    # --------------------------------------------------------
    c = np.zeros(num_vars)
    c[idx["D"]] = -1.0
    c[idx["M"]] = -float(eta_tail)

    # --------------------------------------------------------
    # Bounds and integrality
    # --------------------------------------------------------
    lb = np.full(num_vars, -np.inf)
    ub = np.full(num_vars, np.inf)
    integrality = np.zeros(num_vars, dtype=int)

    lb[idx["bank"]] = -float(bank_bound)
    ub[idx["bank"]] = float(bank_bound)

    for sl in [idx["stock_long"], idx["stock_short"]]:
        lb[sl] = 0.0
        ub[sl] = float(stock_bound)

    for j in range(m):
        for sl in [idx["call_long"][j], idx["call_short"][j], idx["put_long"][j], idx["put_short"][j]]:
            lb[sl] = 0.0
            ub[sl] = float(option_bound)
            if integer_options:
                integrality[sl] = 1

    lb[idx["D"]] = 0.0
    ub[idx["D"]] = np.inf
    lb[idx["M"]] = 0.0
    ub[idx["M"]] = np.inf

    # --------------------------------------------------------
    # Linear constraints
    # --------------------------------------------------------
    rows = []
    lower = []
    upper = []

    # Initial budget equation.
    # Purchases are paid at ask prices and sales receive bid prices.
    row = np.zeros(num_vars)
    row[idx["bank"]] = 1.0
    row[idx["stock_long"]] = market.stock_ask
    row[idx["stock_short"]] = -market.stock_bid

    for j in range(m):
        row[idx["call_long"][j]] = calls[j]["ask"].to_numpy(dtype=float)
        row[idx["call_short"][j]] = -calls[j]["bid"].to_numpy(dtype=float)
        row[idx["put_long"][j]] = puts[j]["ask"].to_numpy(dtype=float)
        row[idx["put_short"][j]] = -puts[j]["bid"].to_numpy(dtype=float)

    rows.append(row)
    lower.append(float(initial_capital))
    upper.append(float(initial_capital))

    # Payoff non-negativity on the finite path grid.
    for ell, v in enumerate(vertices):
        row = np.zeros(num_vars)
        row[idx["bank"]] = df_bar_growth

        for j in range(m):
            sj = v[j]
            row[idx["stock_long"].start + j] = A[j] * sj
            row[idx["stock_short"].start + j] = -A[j] * sj

            if len(calls[j]) > 0:
                Kc = calls[j]["strike"].to_numpy(dtype=float)
                payoff = A[j] * np.maximum(sj - Kc, 0.0)
                row[idx["call_long"][j]] = payoff
                row[idx["call_short"][j]] = -payoff

            if len(puts[j]) > 0:
                Kp = puts[j]["strike"].to_numpy(dtype=float)
                payoff = A[j] * np.maximum(Kp - sj, 0.0)
                row[idx["put_long"][j]] = payoff
                row[idx["put_short"][j]] = -payoff

        row[idx["D"].start + ell] = -1.0
        rows.append(row)
        lower.append(0.0)
        upper.append(np.inf)

    # Nonnegative right-tail slope in each coordinate.
    for j in range(m):
        row = np.zeros(num_vars)
        row[idx["stock_long"].start + j] = 1.0
        row[idx["stock_short"].start + j] = -1.0
        if len(calls[j]) > 0:
            row[idx["call_long"][j]] = 1.0
            row[idx["call_short"][j]] = -1.0
        row[idx["M"].start + j] = -1.0
        rows.append(row)
        lower.append(0.0)
        upper.append(np.inf)

    constraints = LinearConstraint(np.asarray(rows), np.asarray(lower), np.asarray(upper))
    bounds = Bounds(lb, ub)

    options = {}
    if time_limit is not None:
        options["time_limit"] = float(time_limit)
    if mip_rel_gap is not None:
        options["mip_rel_gap"] = float(mip_rel_gap)

    res = milp(
        c=c,
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options=options if options else None,
    )

    if not res.success:
        return {
            "success": False,
            "message": res.message,
            "objective": np.nan,
            "market": market,
            "vertices": vertices,
            "node_sets": node_sets,
        }

    z = res.x
    objective = -float(res.fun)

    result = {
        "success": True,
        "message": res.message,
        "objective": objective,
        "is_arbitrage": objective > tol,
        "z": z,
        "idx": idx,
        "market": market,
        "vertices": vertices,
        "node_sets": node_sets,
        "A": A,
        "T_bar": T_bar,
        "calls": calls,
        "puts": puts,
        "D": z[idx["D"]],
        "M": z[idx["M"]],
    }
    return result


# ============================================================
# Payoff evaluation and reporting
# ============================================================

def payoff_at_path(result: Dict, s: Sequence[float]) -> float:
    """
    Evaluate the accumulated terminal payoff G_theta(s).
    """
    z = result["z"]
    idx = result["idx"]
    market = result["market"]
    calls = result["calls"]
    puts = result["puts"]
    A = result["A"]
    T_bar = result["T_bar"]
    s = np.asarray(s, dtype=float)

    value = z[idx["bank"]] * math.exp(market.r * T_bar)
    m = len(market.maturities)

    for j in range(m):
        stock_net = z[idx["stock_long"].start + j] - z[idx["stock_short"].start + j]
        value += A[j] * stock_net * s[j]

        if len(calls[j]) > 0:
            q = z[idx["call_long"][j]] - z[idx["call_short"][j]]
            K = calls[j]["strike"].to_numpy(dtype=float)
            value += A[j] * np.dot(q, np.maximum(s[j] - K, 0.0))

        if len(puts[j]) > 0:
            q = z[idx["put_long"][j]] - z[idx["put_short"][j]]
            K = puts[j]["strike"].to_numpy(dtype=float)
            value += A[j] * np.dot(q, np.maximum(K - s[j], 0.0))

    return float(value)


def portfolio_tables(result: Dict, position_tol: float = 1e-8):
    """
    Build stock/cash and option-position tables for the optimizer.
    """
    z = result["z"]
    idx = result["idx"]
    market = result["market"]
    calls = result["calls"]
    puts = result["puts"]
    m = len(market.maturities)

    main_rows = [{"instrument": "bank account", "maturity_index": "all", "net_position": z[idx["bank"]]}]
    for j in range(m):
        net_stock = z[idx["stock_long"].start + j] - z[idx["stock_short"].start + j]
        main_rows.append({
            "instrument": "stock liquidation",
            "maturity_index": j + 1,
            "maturity_T": market.maturities[j],
            "net_position": net_stock,
        })
    main_table = pd.DataFrame(main_rows)

    opt_rows = []
    for j in range(m):
        for side, label in [("calls", "call"), ("puts", "put")]:
            df = calls[j] if side == "calls" else puts[j]
            long_sl = idx["call_long"][j] if side == "calls" else idx["put_long"][j]
            short_sl = idx["call_short"][j] if side == "calls" else idx["put_short"][j]
            if len(df) == 0:
                continue
            q = z[long_sl] - z[short_sl]
            for i, row in df.iterrows():
                if abs(q[i]) > position_tol:
                    opt_rows.append({
                        "maturity_index": j + 1,
                        "maturity_T": market.maturities[j],
                        "type": label,
                        "strike": row["strike"],
                        "bid": row["bid"],
                        "ask": row["ask"],
                        "net_position": q[i],
                    })
    option_table = pd.DataFrame(opt_rows)
    return main_table, option_table


def print_multi_maturity_conclusion(result: Dict, tol: float = 1e-8):
    """
    Print a compact arbitrage conclusion and diagnostics.
    """
    print("Solver success:", result.get("success"))
    print("Solver message:", result.get("message"))
    print("Objective value:", result.get("objective"))

    if not result.get("success"):
        return

    print("Finite grid size:", len(result["vertices"]))
    print("Tail margins M_j:", result["M"])
    print("Largest finite payoff margin:", np.max(result["D"]))
    print()

    if result["is_arbitrage"]:
        print("Conclusion: deterministic multi-maturity arbitrage detected.")
    else:
        print("Conclusion: no arbitrage found within the chosen position bounds and numerical tolerance.")

    main_table, option_table = portfolio_tables(result)
    print("\nCash and maturity-specific stock positions:")
    display(main_table)
    print("\nNonzero option positions:")
    if len(option_table) == 0:
        print("No nonzero option positions above the display tolerance.")
    else:
        display(option_table)


def plot_payoff_slices(result: Dict, points: int = 200):
    """
    Plot one-dimensional payoff slices.

    For each maturity j, s_j varies on a horizontal grid while all
    other coordinates are fixed at S0. These plots are diagnostics;
    the actual no-arbitrage certificate is the full Cartesian grid
    plus the tail-slope constraints.
    """
    if not result.get("success"):
        print("No plot is available because the optimization did not succeed.")
        return

    market = result["market"]
    m = len(market.maturities)
    S0 = market.S0

    max_strike = max(max(ns) for ns in result["node_sets"] if len(ns) > 0)
    x_max = max(2.0 * S0, 1.25 * max_strike)
    x_grid = np.linspace(0.0, x_max, points)

    for j in range(m):
        values = []
        for x in x_grid:
            s = np.full(m, S0, dtype=float)
            s[j] = x
            values.append(payoff_at_path(result, s))

        plt.figure(figsize=(8, 4))
        plt.plot(x_grid, values)
        plt.axhline(0.0, linewidth=1.0)
        plt.title(f"Payoff slice for maturity {j + 1}, T={market.maturities[j]:.4f}")
        plt.xlabel(f"s_{j + 1}")
        plt.ylabel("Accumulated payoff at T_bar")
        plt.grid(True, alpha=0.3)
        plt.show()


# ============================================================
# Interactive Colab runner
# ============================================================

def run_synthetic_demo():
    """
    Run the complete workflow on a small synthetic market.
    """
    market = make_synthetic_multi_maturity_market(inject_violation=True)
    result = build_and_solve_multi_maturity_arbitrage(
        market,
        initial_capital=0.0,
        stock_bound=10.0,
        option_bound=10.0,
        integer_options=False,
        max_grid_points=20000,
    )
    print_multi_maturity_conclusion(result)
    plot_payoff_slices(result)
    return result


def run_yahoo_interactive_analysis():
    """
    Interactive Yahoo Finance workflow for Colab.

    The user chooses a ticker and several expiration dates. The
    helper keeps a small number of near-the-money strikes per side
    so that the Cartesian multi-maturity grid remains manageable.
    """
    ticker = input("Ticker symbol, for example AAPL: ").strip().upper()
    expirations = show_yahoo_expirations(ticker)

    raw = input("Enter expiration numbers separated by commas, for example 1,2,3: ").strip()
    selected = []
    for token in raw.split(","):
        k = int(token.strip())
        selected.append(expirations[k - 1])

    r = float(input("Continuously compounded risk-free rate, for example 0.03: ").strip() or "0.03")
    max_side = int(input("Max options per side and maturity, for example 5 or 6: ").strip() or "6")
    pct = float(input("Bid-ask perturbation percent, for example 0 or 1: ").strip() or "0")

    market = fetch_yahoo_multi_maturity_market(
        ticker=ticker,
        expiration_dates=selected,
        r=r,
        max_options_per_side=max_side,
    )
    market = apply_transaction_cost_perturbation(market, perturbation_percent=pct)

    result = build_and_solve_multi_maturity_arbitrage(
        market,
        initial_capital=0.0,
        stock_bound=10.0,
        option_bound=10.0,
        integer_options=False,
        max_grid_points=20000,
    )
    print_multi_maturity_conclusion(result)
    plot_payoff_slices(result)
    return result


## Cell 3 — Run the synthetic example

The synthetic example is small enough to run quickly in Colab. For live Yahoo Finance data, call `run_yahoo_interactive_analysis()` instead.

In [ ]:
# Run the synthetic multi-maturity arbitrage demo.
# For live Yahoo data, run: results = run_yahoo_interactive_analysis()
results = run_synthetic_demo()
